# 6 — Multi-Agent Router Graph

A **router LLM** reads each user message, classifies it as PRODUCT / ORDER / SMALLTALK / END, then dispatches to the right specialist sub-agent. Each specialist has its own tools and memory.

**What you'll learn**
- Router pattern: LLM returns a single keyword → `add_conditional_edges` maps it to a node
- `functools.partial` — binding a specialist agent into a graph node
- Nested MemorySavers — each sub-agent keeps its own conversation history
- How to build supervisor-style graphs without hardcoded routing logic
- Why specialist agents with focused prompts outperform one giant agent with all tools

**Companion examples:** 5-react-agent-lg (two-agent critic loop), 25-adaptive-rag (query-routing for retrieval)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. The router pattern explained ───────────────────────────────────────────
# The key mechanism: add_conditional_edges maps the return value of find_route
# to a node name. The router LLM is trained to return ONLY one keyword.
#
# graph.add_conditional_edges(
#     "Router",
#     find_route,          # reads last message content -> string
#     {
#         "PRODUCT": "Product_Agent",   # specialist: product specs + pricing
#         "ORDER":   "Orders_Agent",    # specialist: order lookup + updates
#         "SMALLTALK": "Small_Talk",    # lightweight: greeting handler
#         "END":     END,               # no action needed
#     }
# )
#
# Each specialist runs in isolation — they don't see each other's messages.
# The RouterAgentState accumulates ALL messages (operator.add),
# but each sub-agent only sees its own internal thread via MemorySaver.
print("Router dispatch:  'SpectraBook features?' -> PRODUCT -> Product_Agent")
print("                  'Status of ORD-7311?'   -> ORDER   -> Orders_Agent")
print("                  'Hi, how are you?'       -> SMALLTALK -> Small_Talk")

In [ ]:
# ── 4. Inline data + specialist tools ─────────────────────────────────────────
# Original script loads Laptop pricing.csv, Laptop Orders.csv, and a product PDF.
# We replace all three with inline dicts — same tool interfaces, no file I/O.
import functools
import operator
import uuid
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.prebuilt import create_react_agent

PRODUCT_CATALOG = {
    "SpectraBook": {
        "cpu": "Intel Core Ultra 7",
        "ram": "16GB DDR5",
        "storage": "512GB NVMe SSD",
        "display": "14-inch IPS 2560x1600",
        "battery": "72Wh up to 12h",
        "price": 1299,
    },
    "AlphaBook Pro": {
        "cpu": "AMD Ryzen 9 7940H",
        "ram": "32GB DDR5",
        "storage": "1TB NVMe SSD",
        "display": "15.6-inch OLED 2880x1800",
        "battery": "86Wh up to 10h",
        "price": 1799,
    },
    "NovaPad": {
        "cpu": "Apple M3",
        "ram": "8GB unified",
        "storage": "256GB SSD",
        "display": "13.3-inch Retina",
        "battery": "58Wh up to 18h",
        "price": 999,
    },
}

ORDERS = {
    "ORD-7311": {"product": "SpectraBook", "quantity": 2, "delivery": "2025-07-15"},
    "ORD-8842": {"product": "AlphaBook Pro", "quantity": 1, "delivery": "2025-07-20"},
}


@tool
def get_product_features(product_name: str) -> str:
    """Returns detailed specs and features for a laptop by name."""
    for key, val in PRODUCT_CATALOG.items():
        if product_name.lower() in key.lower():
            specs = ", ".join(f"{k}: {v}" for k, v in val.items() if k != "price")
            return f"{key} — {specs}"
    return "Product not found"


@tool
def get_laptop_price(product_name: str) -> str:
    """Returns the price of a laptop by name."""
    for key, val in PRODUCT_CATALOG.items():
        if product_name.lower() in key.lower():
            return f"{key} costs USD {val['price']}"
    return "Product not found"


@tool
def get_order_details(order_id: str) -> str:
    """Returns details about a laptop order given its order ID."""
    order = ORDERS.get(order_id)
    return str(order) if order else f"Order {order_id} not found"


@tool
def update_quantity(order_id: str, new_quantity: int) -> str:
    """Updates the quantity for an order given its order ID and new quantity."""
    if order_id not in ORDERS:
        return f"Order {order_id} not found"
    ORDERS[order_id]["quantity"] = new_quantity
    return f"Updated order {order_id} quantity to {new_quantity}"


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

product_agent = create_react_agent(
    model=llm,
    tools=[get_product_features, get_laptop_price],
    prompt=SystemMessage("Answer laptop product questions using ONLY the tools. Be concise."),
    checkpointer=MemorySaver(),
)

orders_agent = create_react_agent(
    model=llm,
    tools=[get_order_details, update_quantity],
    prompt=SystemMessage("Handle order queries using ONLY the tools. Be concise."),
    checkpointer=MemorySaver(),
)

print("Specialist agents ready.")

In [ ]:
# ── 5. Build the RouterAgent graph ────────────────────────────────────────────
# functools.partial binds the agent into the node function signature.
# Each sub-agent gets its own thread_id so their MemorySavers don't mix.


class RouterAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


def agent_node(state: RouterAgentState, agent) -> dict:
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = agent.invoke(state, config)
    return {"messages": [AIMessage(result["messages"][-1].content)]}


product_node = functools.partial(agent_node, agent=product_agent)
orders_node_fn = functools.partial(agent_node, agent=orders_agent)

ROUTER_PROMPT = (
    "You are a router. Classify the query and respond with ONLY ONE word:\n"
    "PRODUCT — questions about laptop specs, features, or pricing\n"
    "ORDER — questions about orders: status, details, quantity updates\n"
    "SMALLTALK — greetings, goodbyes, general small talk\n"
    "END — anything else"
)


def router_llm(state: RouterAgentState) -> dict:
    messages = [SystemMessage(ROUTER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"  [router -> {result.content.strip()}]")
    return {"messages": [result]}


def find_route(state: RouterAgentState) -> str:
    return state["messages"][-1].content.strip().upper()


def smalltalk(state: RouterAgentState) -> dict:
    messages = [
        SystemMessage(
            "Respond professionally to greetings. "
            "Mention you can help with laptop products and orders."
        )
    ] + state["messages"]
    return {"messages": [llm.invoke(messages)]}


graph = StateGraph(RouterAgentState)
graph.add_node("Router", router_llm)
graph.add_node("Product_Agent", product_node)
graph.add_node("Orders_Agent", orders_node_fn)
graph.add_node("Small_Talk", smalltalk)
graph.add_conditional_edges(
    "Router",
    find_route,
    {"PRODUCT": "Product_Agent", "ORDER": "Orders_Agent", "SMALLTALK": "Small_Talk", "END": END},
)
graph.add_edge("Product_Agent", END)
graph.add_edge("Orders_Agent", END)
graph.add_edge("Small_Talk", END)
graph.set_entry_point("Router")
router_app = graph.compile()

print("Router graph compiled — nodes: Router, Product_Agent, Orders_Agent, Small_Talk")

In [ ]:
# ── 6. Demo the routing ────────────────────────────────────────────────────────
config = {"configurable": {"thread_id": "demo"}}

queries = [
    "Hi, how are you?",
    "Tell me about the SpectraBook features",
    "What is the status of order ORD-7311?",
    "Update that order to 3 units",
    "How much does the AlphaBook Pro cost?",
]

for q in queries:
    print(f"\nUSER: {q}")
    r = router_app.invoke({"messages": [HumanMessage(q)]}, config)
    print(f"AGENT: {r['messages'][-1].content}")

## Exercises

**Exercise 1 — Add a new route:** Add a `COMPARE` route that handles queries like "compare SpectraBook vs AlphaBook Pro." Create a compare tool and a new specialist node.

**Exercise 2 — What happens on bad routing?** Change a product query so the router misclassifies it as ORDER. What does the Orders_Agent return? How would you add a fallback?

**Exercise 3 — Sub-agent memory:** Right now each sub-agent call gets a fresh `uuid.uuid4()` thread_id. Change this so multi-turn conversations with the same specialist persist. Hint: pass the outer `thread_id` through to the sub-agent's config.

**Exercise 4 — Class vs. prebuilt:** The original script uses a custom `OrdersAgent` class instead of `create_react_agent`. Read `src/agent/orders.py`. What extra control does the custom class give? When would you choose it over the prebuilt?

## What's next

- **5-react-agent-lg** — two-agent critic loop without a router
- **25-adaptive-rag** — same routing idea applied to retrieval strategies
- **19-multi-agent-notebook** — multi-agent collaboration with shared state